# Fine-tuning clássico — só a última camada

Experimento **baseline** do projeto *Transfer learning em domínio distante*.

**Definição de fine-tuning clássico usada aqui:** a ResNet-18 pré-treinada em ImageNet
(`microsoft/resnet-18`, Hugging Face) é **totalmente congelada**, a camada final
(1000 classes → 10 classes do EuroSAT) é **substituída por uma nova** e **somente ela é treinada**.

Tudo o que for variado depois (descongelar `layer4`, rede inteira, LR diferencial, treino do zero)
deve reutilizar **exatamente** o mesmo split, semente, épocas e transforms deste notebook.
O split e os resultados são salvos em `results/` para comparação futura.

In [ ]:
!pip -q install transformers
import json, time, random, os
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import EuroSAT
from transformers import ResNetForImageClassification
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Configuração (não alterar entre experimentos que serão comparados)

In [ ]:
CFG = dict(
    modelo="microsoft/resnet-18",
    regime="fine-tuning classico (so a ultima camada)",
    seed=42,
    n_train=2000, n_val=1000, n_test=5000,   # split estratificado fixo
    epocas=15, batch=64,
    otimizador="Adam", lr=1e-3, weight_decay=0.0,
    img_size=224,
)
OUT = "results"; os.makedirs(OUT, exist_ok=True)
TAG = "ft_classico"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])

## 2. Dados e split estratificado (salvo em disco)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet, mesmos do pre-treino
S = CFG["img_size"]
tf_eval  = transforms.Compose([transforms.Resize(S), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_train = transforms.Compose([transforms.Resize(S),
                               transforms.RandomHorizontalFlip(),
                               transforms.RandomCrop(S, padding=16, padding_mode="reflect"),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

base = EuroSAT(root="./data", download=True)
y = np.array(base.targets); classes = base.classes

SPLIT_PATH = f"{OUT}/split.json"
if os.path.exists(SPLIT_PATH):                      # reutiliza o split de execucoes anteriores
    sp = json.load(open(SPLIT_PATH))
else:
    rng = np.random.RandomState(CFG["seed"])
    tr, va, te = [], [], []
    per = lambda n: n // len(classes)
    for c in range(len(classes)):
        idx = rng.permutation(np.where(y == c)[0])
        a, b = per(CFG["n_train"]), per(CFG["n_val"])
        tr += idx[:a].tolist(); va += idx[a:a+b].tolist(); te += idx[a+b:a+b+per(CFG["n_test"])].tolist()
    sp = dict(train=tr, val=va, test=te); json.dump(sp, open(SPLIT_PATH, "w"))

class Wrap(torch.utils.data.Dataset):
    def __init__(s, idx, tf): s.idx, s.tf = idx, tf
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        img, t = base[s.idx[i]]; return s.tf(img), t

mk = lambda idx, tf, sh: DataLoader(Wrap(idx, tf), batch_size=CFG["batch"], shuffle=sh, num_workers=2, pin_memory=True)
dl_train = mk(sp["train"], tf_train, True)
dl_train_eval = mk(sp["train"], tf_eval, False)    # treino SEM augmentation, para medir o gap
dl_val, dl_test = mk(sp["val"], tf_eval, False), mk(sp["test"], tf_eval, False)
print({k: len(v) for k, v in sp.items()}, "| classes:", len(classes))
print("treino por classe:", np.bincount(y[sp["train"]]))
assert not (set(sp["train"]) & set(sp["test"])) and not (set(sp["train"]) & set(sp["val"]))

## 3. Modelo: tudo congelado, só a nova camada final treina

In [ ]:
model = ResNetForImageClassification.from_pretrained(CFG["modelo"])
print("cabeca original:", model.classifier)

model.classifier[1] = nn.Linear(model.config.hidden_sizes[-1], len(classes))   # 512 -> 10, inicializacao nova
model.config.num_labels = len(classes)

for p in model.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True
model.to(device)

n_tot = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parametros treinaveis: {n_tr:,} de {n_tot:,} ({n_tr/n_tot:.3%})")
CFG["params_treinaveis"], CFG["params_total"] = n_tr, n_tot

def set_mode(m):
    m.eval()                       # backbone em eval: BatchNorm usa estatisticas do ImageNet, nao atualiza
    m.classifier.train()           # (a cabeca nao tem BN/dropout, mas mantem o padrao)

crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.classifier.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

## 4. Treino

In [ ]:
@torch.no_grad()
def avaliar(dl, ret_pred=False):
    model.eval(); tot = corr = 0; loss = 0.0; P, T = [], []
    for x, t in dl:
        x, t = x.to(device), t.to(device)
        lo = model(pixel_values=x).logits
        loss += crit(lo, t).item() * len(t); pr = lo.argmax(1)
        corr += (pr == t).sum().item(); tot += len(t); P.append(pr.cpu()); T.append(t.cpu())
    r = (corr / tot, loss / tot)
    return (*r, torch.cat(P).numpy(), torch.cat(T).numpy()) if ret_pred else r

hist = dict(train_loss=[], train_acc_aug=[], val_acc=[], val_loss=[])
best = dict(val_acc=-1, epoca=0, state=None)
t0 = time.time()
for ep in range(1, CFG["epocas"] + 1):
    set_mode(model); L = A = N = 0
    for x, t in dl_train:
        x, t = x.to(device), t.to(device)
        lo = model(pixel_values=x).logits; loss = crit(lo, t)
        opt.zero_grad(); loss.backward(); opt.step()
        L += loss.item() * len(t); A += (lo.argmax(1) == t).sum().item(); N += len(t)
    va, vl = avaliar(dl_val)
    hist["train_loss"].append(L / N); hist["train_acc_aug"].append(A / N)
    hist["val_acc"].append(va); hist["val_loss"].append(vl)
    if va > best["val_acc"]:
        best = dict(val_acc=va, epoca=ep, state={k: v.detach().clone() for k, v in model.classifier.state_dict().items()})
    print(f"ep {ep:2d} | loss {L/N:.4f} | acc treino(aug) {A/N:.3f} | acc val {va:.3f}")
tempo = time.time() - t0
print(f"tempo de treino: {tempo:.0f}s | melhor epoca (val): {best['epoca']}")

## 5. Resultados (checkpoint da melhor época na validação)

In [ ]:
model.classifier.load_state_dict(best["state"])
acc_tr, _ = avaliar(dl_train_eval)
acc_va, _ = avaliar(dl_val)
acc_te, loss_te, pred, true = avaliar(dl_test, ret_pred=True)
cm = confusion_matrix(true, pred)
por_classe = {c: float(cm[i, i] / cm[i].sum()) for i, c in enumerate(classes)}

print(f"acuracia treino (sem aug): {acc_tr:.4f}")
print(f"acuracia validacao       : {acc_va:.4f}")
print(f"acuracia TESTE           : {acc_te:.4f}")
print(f"gap treino - teste       : {acc_tr - acc_te:+.4f}")
print("\nacuracia por classe (teste):")
for c, v in sorted(por_classe.items(), key=lambda kv: kv[1]): print(f"  {c:22s} {v:.3f}")

res = dict(config=CFG, melhor_epoca=best["epoca"], tempo_treino_s=tempo,
           acc_treino=acc_tr, acc_val=acc_va, acc_teste=acc_te, loss_teste=loss_te,
           gap=acc_tr - acc_te, acc_por_classe=por_classe,
           confusion_matrix=cm.tolist(), classes=classes, historico=hist)
json.dump(res, open(f"{OUT}/{TAG}.json", "w"), indent=2)
print(f"\nsalvo em {OUT}/{TAG}.json")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ep = range(1, len(hist["val_acc"]) + 1)
ax[0].plot(ep, hist["train_loss"], label="treino"); ax[0].plot(ep, hist["val_loss"], label="validação")
ax[0].set(title="Loss", xlabel="época"); ax[0].legend()
ax[1].plot(ep, hist["train_acc_aug"], label="treino (com aug)"); ax[1].plot(ep, hist["val_acc"], label="validação")
ax[1].axvline(best["epoca"], ls=":", c="gray"); ax[1].set(title="Acurácia", xlabel="época"); ax[1].legend()
im = ax[2].imshow(cm, cmap="Blues"); ax[2].set(title=f"Matriz de confusão (teste) — acc {acc_te:.3f}")
ax[2].set_xticks(range(10)); ax[2].set_yticks(range(10))
ax[2].set_xticklabels(classes, rotation=75, fontsize=7); ax[2].set_yticklabels(classes, fontsize=7)
for i in range(10):
    for j in range(10): ax[2].text(j, i, cm[i, j], ha="center", va="center", fontsize=6, color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig(f"{OUT}/{TAG}.png", dpi=150); plt.show()

## 6. Para comparar depois

Os próximos experimentos devem carregar `results/split.json` (já reutilizado automaticamente) e salvar
`results/<nome>.json` com a mesma estrutura. A comparação é feita em `acc_teste`, `gap`,
`acc_por_classe` e `historico`. **Não** use o teste para escolher época ou hiperparâmetros — só a validação.